# Prueba del tablero sin servidor
**Tarea 5 · Soporte 5.** Registro de las pruebas de `app.py`: se llama a cada callback directamente (como la "prueba sin servidor" de la sesión 7) y se verifica que las cifras coincidan con el modelado.

Para ver el tablero: `python app.py` y abrir http://127.0.0.1:8050

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import app as a
print("Créditos de octubre:", a.N_OCT, "| impago real:", a.pct(a.IMPAGO_OCT), "| meta:", a.pct(a.LIMITE))
a.METRICAS.round(3)

Créditos de octubre: 98364 | impago real: 23,51 % | meta: 21,71 %


,AUC,KS,PR-AUC
Gradient boosting,0.648,0.218,0.345
Regresión logística,0.639,0.204,0.338
Random Forest,0.635,0.197,0.334
Regla del buró,0.554,0.074,0.263


## 1 · Pestaña Riesgo: efecto del % de aprobación (CN1 y CN2)

In [2]:
filas = []
for modelo in ["Gradient boosting", "Regresión logística", "Regla del buró"]:
    for tasa in [85, 88, 90, 95]:
        k = a.kpis(a.aplicar_corte(modelo, tasa / 100))
        filas.append({"modelo": modelo, "% aprobado": tasa, "impago aprobados": a.pct(k["impago_aprob"]),
                      "cumple CN2": k["impago_aprob"] < a.LIMITE, "impagos evitados": k["evitados"],
                      "buenos rechazados": k["buenos_rech"], "monto evitado": a.millones(k["monto_evitado"])})
pd.DataFrame(filas)

,modelo,% aprobado,impago aprobados,cumple CN2,impagos evitados,buenos rechazados,monto evitado
0,Gradient boosting,85,"20,93 %",True,5634,9121,"341,0 M"
1,Gradient boosting,88,"21,40 %",True,4605,7199,"278,4 M"
2,Gradient boosting,90,"21,70 %",True,3923,5914,"237,1 M"
3,Gradient boosting,95,"22,47 %",False,2134,2785,"129,7 M"
4,Regresión logística,85,"20,98 %",True,5591,9164,"340,4 M"
5,Regresión logística,88,"21,42 %",True,4585,7219,"279,6 M"
6,Regresión logística,90,"21,72 %",False,3905,5932,"238,8 M"
7,Regresión logística,95,"22,48 %",False,2123,2796,"130,8 M"
8,Regla del buró,85,"22,25 %",False,4531,10224,"266,3 M"
9,Regla del buró,88,"22,49 %",False,3660,8144,"215,2 M"


In [3]:
salida = a.actualizar_riesgo("Gradient boosting", 90, "Todas")
print("Tarjetas:", [s if isinstance(s, str) else s.children for s in salida[:6]])
salida[6]   # curva aprobación vs impago

Tarjetas: ['90,0 %', '21,70 %', '3 923', '5 914', '237,1 M', '0.648']


## 2 · Pestaña Ventas: modelo frente a la regla del buró (PE1)

In [4]:
v = a.actualizar_ventas(90)
print("Aprobados:", v[0], "| prob. de pago media:", v[1], "| impago aprobados:", v[2].children)
print("Máximo aprobable con impago < 21,71 %:", {m: a.num(n) for m, n in a.MAXIMOS.items()})
pd.DataFrame(v[5]).head(10)

Aprobados: 88 527 (90 %) | prob. de pago media: 82,1 % | impago aprobados: 21,70 %
Máximo aprobable con impago < 21,71 %: {'Gradient boosting': '88 666', 'Regresión logística': '88 488', 'Random Forest': '87 790', 'Regla del buró': '34 467'}


,BRANCH_ID,creditos,prob_pago,impago_real,pct_aprobado
0,152,2731,88.2,15.5,99.7
1,17,637,87.7,16.5,99.8
2,66,205,86.5,20.0,99.5
3,19,2586,86.1,18.1,98.7
4,1,3419,86.0,16.1,99.6
5,104,556,85.9,18.0,99.3
6,8,1542,85.3,13.6,98.0
7,162,1262,84.3,18.2,98.4
8,2,5301,84.2,20.1,98.9
9,135,1343,84.1,20.1,98.1


## 3 · Pestaña Evaluar cliente: 3 casos de prueba

In [5]:
casos = {
    "Bajo riesgo (asalariado, LTV 73 %, sin historial)":
        ("1990-05-10", "Salaried", 55000, 75000, "67", "86", "Sin historial", 0, 0, 0, 0, 0, 0, 0, 0, 0),
    "Alto riesgo (joven, LTV 95 %, buró muy alto, mora)":
        ("1999-05-10", "Self employed", 70000, 74000, "251", "48", "Muy alto", 5, 4, 3, 200000, 1, 2, 4, 2, 2),
    "Datos inválidos (menor de edad, LTV > 100 %)":
        ("2010-05-10", "Salaried", 90000, 50000, "67", "86", "Bajo", 0, 0, 0, 0, 0, 0, 0, 0, 0),
}
for nombre, valores in casos.items():
    fila, edad, ltv, alertas = a.construir_cliente(*valores)
    if any("LTV" in x for x in alertas):
        print(f"{nombre}: NO se calcula -> {alertas}")
        continue
    p = float(a.MODELO_VENTAS.predict_proba(fila)[:, 1][0])
    print(f"{nombre}: edad {edad}, LTV {ltv:.1f} % -> P(pago) {a.pct(1 - p, 1)} -> {a.decision(p, 'ventas')[0]}")

Bajo riesgo (asalariado, LTV 73 %, sin historial): edad 36, LTV 73.3 % -> P(pago) 88,3 % -> Aprobar


Alto riesgo (joven, LTV 95 %, buró muy alto, mora): edad 27, LTV 94.6 % -> P(pago) 37,6 % -> Rechazar
Datos inválidos (menor de edad, LTV > 100 %): NO se calcula -> ['El cliente tiene 16 años: es menor de edad.', 'El LTV debe estar entre 0 y 100 %: revisa el monto y el valor del vehículo.']


In [6]:
resultado, figura = a.evaluar_cliente(1, *casos["Alto riesgo (joven, LTV 95 %, buró muy alto, mora)"])
figura   # explicación: variables que más suben o bajan el riesgo

## 4 · Resumen ejecutivo (conclusiones calculadas)

In [7]:
for c in a.CONCLUSIONES:
    print("•", c)

• Aprobando el 90% de las solicitudes de octubre, el impago de los aprobados baja de 23,51 % a 21,70 % con gradient boosting (meta: menos de 21,71 %).
• Con el mismo límite de impago, el modelo puede aprobar 88 666 créditos y la regla del buró solo 34 467: 54 199 clientes viables más.
• Rechazar el 10% más riesgoso evita 3 923 impagos y 237,1 M desembolsados a clientes que no pagaron; el costo es rechazar 5 914 buenos pagadores.
• Las variables que más pesan son: LTV (% financiado), Sucursal, Categoría del buró.
